In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, roc_auc_score
from matplotlib.colors import LinearSegmentedColormap

# ==========================================
# 1. Configuration & Data Loading
# ==========================================
def load_data():
    try:
        df1 = pd.read_csv('D:/R-work/1-age-gender.csv')
        df2 = pd.read_csv('D:/R-work/2-age-gender.csv')
        df3 = pd.read_csv('D:/R-work/3-age-gender.csv')
        return df1, df2, df3
    except FileNotFoundError:
        print("Error: Please ensure CSV files are in the current directory.")
        return None, None, None

def get_raw_group(pid):

    if not isinstance(pid, str): return None
    parts = pid.split('.')
    for p in parts:
        if p == 'G': return 'G'
        if p == 'SE': return 'NM' 
        if p == 'HW': return 'HW'
        if p == 'H': return 'HW'
    return None

def process_batch_subset_normalize(df, positive_groups, negative_groups, protein_cols):

    d = df.copy()
    d['Group'] = d['Protein.Ids'].apply(get_raw_group)
    
    # Filter
    all_groups = positive_groups + negative_groups
    d = d[d['Group'].isin(all_groups)].copy()
    
    if len(d) == 0:
        return d
    
    # Normalize
    cols_to_scale = [c for c in protein_cols if c in d.columns]
    if cols_to_scale:
        scaler = StandardScaler()
        d[cols_to_scale] = scaler.fit_transform(d[cols_to_scale])
        
    return d

def calculate_auc_ci(y_true, y_pred_prob, n_bootstraps=1000, alpha=0.05, random_state=42):
    rng = np.random.RandomState(random_state)
    bootstrapped_scores = []
    
    y_true = np.array(y_true)
    y_pred_prob = np.array(y_pred_prob)
    
    for i in range(n_bootstraps):
        indices = rng.randint(0, len(y_pred_prob), len(y_pred_prob))
        if len(np.unique(y_true[indices])) < 2:
            continue
        score = roc_auc_score(y_true[indices], y_pred_prob[indices])
        bootstrapped_scores.append(score)
        
    sorted_scores = np.array(bootstrapped_scores)
    sorted_scores.sort()
    
    lower_bound = np.percentile(sorted_scores, (alpha / 2) * 100)
    upper_bound = np.percentile(sorted_scores, (1.0 - alpha / 2) * 100)
    
    return lower_bound, upper_bound

df1, df2, df3 = load_data()

if df1 is not None:

    common_cols = set(df1.columns).intersection(df2.columns).intersection(df3.columns)
    meta_cols = ['Protein.Ids', 'Age', 'Gender']
    protein_cols = [c for c in common_cols if c not in meta_cols and 'LOG2' not in c]
    protein_cols.sort()
    
    pos_groups = ['G', 'NM'] 
    neg_groups = ['HW']      
    

    cmap_female = LinearSegmentedColormap.from_list("female_brown", ["#ffffff", "#A52A2A"], N=100)
    roc_color_female = '#d62728' 
    
    cmap_male = LinearSegmentedColormap.from_list("male_navy", ["#ffffff", "#000080"], N=100) 
    roc_color_male = '#000080' 
    

    gender_map = {1: 'Male', 0: 'Female'}
    
    for g_code, g_name in gender_map.items():
        print(f"--- Processing {g_name} ---")
        
        if g_name == 'Male':
            current_cmap = cmap_male
            current_roc_color = roc_color_male
        else:
            current_cmap = cmap_female
            current_roc_color = roc_color_female
        

        df1_sub = process_batch_subset_normalize(df1, pos_groups, neg_groups, protein_cols)
        df2_sub = process_batch_subset_normalize(df2, pos_groups, neg_groups, protein_cols)
        df3_sub = process_batch_subset_normalize(df3, pos_groups, neg_groups, protein_cols)
        

        train_df = pd.concat([df1_sub, df2_sub], ignore_index=True)
        test_df = df3_sub
        

        tr = train_df[train_df['Gender'] == g_code].copy()
        te = test_df[test_df['Gender'] == g_code].copy()
        
        if len(tr) < 5 or len(te) < 3:
            print(f"Skipping {g_name}: Not enough data.")
            continue
            

        tr['Target'] = tr['Group'].apply(lambda x: 1 if x in pos_groups else 0)
        te['Target'] = te['Group'].apply(lambda x: 1 if x in pos_groups else 0)
        
        y_train = tr['Target'].astype(int)
        y_test = te['Target'].astype(int)
        
        X_train = tr[protein_cols].fillna(0)
        X_test = te[protein_cols].fillna(0)
        

        selector = SelectKBest(f_classif, k=10)
        selector.fit(X_train, y_train)
        
        X_train_sel = selector.transform(X_train)
        X_test_sel = selector.transform(X_test)
        

        clf = LogisticRegression(max_iter=5000, random_state=45)
        clf.fit(X_train_sel, y_train)
        

        y_pred = clf.predict(X_test_sel)
        y_prob = clf.predict_proba(X_test_sel)[:, 1]

        fig, axes = plt.subplots(1, 2, figsize=(15, 6)) 
        

        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc_val = auc(fpr, tpr)
        
        ci_lower, ci_upper = calculate_auc_ci(y_test, y_prob, n_bootstraps=1000)
        legend_text = f'AUC = {auc_val:.2f} (95% CI: {ci_lower:.2f}-{ci_upper:.2f})'
        
        axes[0].plot(fpr, tpr, lw=6, color=current_roc_color, label=legend_text)
        axes[0].plot([0, 1], [0, 1], 'k--', lw=3, alpha=0.5)
        axes[0].set_xlim([-0.02, 1.0])
        axes[0].set_ylim([0.0, 1.05])
        
        axes[0].set_xlabel('False Positive Rate', fontsize=24, fontweight='bold', labelpad=10)
        axes[0].set_ylabel('True Positive Rate', fontsize=24, fontweight='bold', labelpad=10)
        axes[0].set_title(f'{g_name}: ROC Curve', fontsize=24, fontweight='bold', pad=15)
        axes[0].tick_params(axis='both', which='major', labelsize=24)
        axes[0].legend(loc="lower right", fontsize=16, frameon=True) 
        axes[0].grid(alpha=0.3, linestyle='--')
        for label in axes[0].get_xticklabels() + axes[0].get_yticklabels():
            label.set_fontweight('bold')
        for spine in axes[0].spines.values():
            spine.set_linewidth(2.5)
            spine.set_color('black')
        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['H', 'LUAD'])
        
        disp.plot(cmap=current_cmap, ax=axes[1], values_format='d', colorbar=False)
        
        max_val = cm.max()
        for text in disp.text_.ravel():
            current_val = int(text.get_text())
            if current_val == max_val:
                text.set_color('white')
            else:
                text.set_color('black')
                
            text.set_fontsize(24)
            text.set_weight('bold')
            
        axes[1].set_title(f'{g_name}: Confusion Matrix', fontsize=24, fontweight='bold', pad=15)
        axes[1].set_xlabel('Predicted Label', fontsize=24, fontweight='bold', labelpad=10)
        axes[1].set_ylabel('True Label', fontsize=24, fontweight='bold', labelpad=10)
        axes[1].tick_params(axis='both', which='major', labelsize=24)
        for label in axes[1].get_xticklabels() + axes[1].get_yticklabels():
            label.set_fontweight('bold')
        for spine in axes[1].spines.values():
            spine.set_linewidth(2.5)
            spine.set_color('black')
        plt.tight_layout()
        outfile = f'analysis_{g_name}_LUAD_H.png'
        plt.savefig(outfile, dpi=600, bbox_inches='tight')
        plt.show()
        print(f"Successfully saved: {outfile}")